# E-Commerce Sales & Customer Analytics

**Business goal:** analyze sales performance, customer value, product performance, and order operations for an e-commerce company.

**Tools:** Python, Pandas, NumPy, Matplotlib.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path('../data')
customers = pd.read_csv(DATA/'customers.csv', parse_dates=['signup_date'])
products = pd.read_csv(DATA/'products.csv')
orders = pd.read_csv(DATA/'orders.csv', parse_dates=['order_date'])
items = pd.read_csv(DATA/'order_items.csv')

print(customers.shape, products.shape, orders.shape, items.shape)

# Data Quality

Check duplicates, missing values, and data types before analysis.

In [ ]:
for name, df in {'customers':customers,'products':products,'orders':orders,'order_items':items}.items():
    print('\n', name)
    print(df.info())
    print('duplicates:', df.duplicated().sum())
    print('missing values:\n', df.isna().sum())

# Build the analysis dataset

Join the transactional data with customer and product attributes.

In [ ]:
df = (items.merge(orders, on='order_id', how='left')
           .merge(products, on='product_id', how='left')
           .merge(customers, on='customer_id', how='left'))
completed = df[df['status'].isin(['Completed','Shipped'])].copy()
df.head()

# Executive KPIs

In [ ]:
kpis = {
    'Revenue': completed.sales_amount.sum(),
    'Profit': completed.profit_amount.sum(),
    'Orders': completed.order_id.nunique(),
    'Customers': completed.customer_id.nunique(),
    'Units Sold': completed.quantity.sum()
}
kpis['Average Order Value'] = kpis['Revenue']/kpis['Orders']
pd.Series(kpis).round(2)

# Monthly Revenue Trend

In [ ]:
monthly = (completed.assign(month=completed.order_date.dt.to_period('M').astype(str))
           .groupby('month', as_index=False)
           .agg(revenue=('sales_amount','sum'), profit=('profit_amount','sum')))
plt.figure(figsize=(12,5))
plt.plot(monthly.month, monthly.revenue, marker='o')
plt.xticks(rotation=60)
plt.title('Monthly Revenue Trend')
plt.ylabel('Revenue')
plt.tight_layout()
plt.show()

# Product & Category Performance

In [ ]:
category = (completed.groupby('category', as_index=False)
            .agg(revenue=('sales_amount','sum'), profit=('profit_amount','sum'), units=('quantity','sum'))
            .sort_values('revenue', ascending=False))
category

# Top 10 Customers

In [ ]:
(completed.groupby(['customer_id','first_name','last_name'], as_index=False)
 .agg(revenue=('sales_amount','sum'), orders=('order_id','nunique'), profit=('profit_amount','sum'))
 .sort_values('revenue', ascending=False).head(10))